In [1]:
import pandas as pd
import numpy as np
import random
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold

from data_preprocessing.get_stnthetic_data import *

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(100)  # ✅ Set your global seed here

## Load dataset

#### Synthetic data

In [2]:
train_set_bs, test_set_bs, train_set_ps, test_set_ps, train_set_po, test_set_po = get_synthetic_data("/Users/ccy/Documents/CMU/Spring2025/42687 Projects in Biomedical AI/Final Project/Explainable Deep Learning on Multimodal Patient Data for Predicting Immunotherapy Outcomes in Metastatic Non-small Cell Lung Cancer/Synthetic data/PCA/rfe_50.csv",train_set_ratio=0.7, k=3)

# Change the input here
def input(train):
    X = train.iloc[:, :-1].values
    print(X.shape)
    
    # Standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    target_col = train.columns[-1]
    print(target_col)
    y = train[target_col].values
    print("Label distribution:", Counter(y))
    return X_scaled, y

X_scaled, y = input(train_set_bs)

(400, 50)
Best response
Label distribution: Counter({1.0: 200, 0.0: 200})


### Define MLP

In [3]:
# Define MLP Model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_dims=[16, 8, 4], dropout=0.3):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_size
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.model(x))


### Training

In [4]:
def cross_validate_models(X_scaled, y, model_configs, num_epochs=100, batch_size=32):
    results = []

    for config in model_configs:
        print(f"\n🔍 Testing model config: {config}")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        train_f1_scores, val_f1_scores = [], []
        train_acc_scores, val_acc_scores = [], []


        for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
            X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
            X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
            y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

            train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)

            model = MLP(input_size=X_scaled.shape[1], hidden_dims=config["hidden_dims"], dropout=config["dropout"])
            criterion = nn.BCELoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
            #scheduler = StepLR(optimizer, step_size=50, gamma=0.9)

            all_train_preds, all_train_labels = [], []
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    outputs = model(inputs)
                    preds = (outputs > 0.5).float()
                    all_train_preds.extend(preds.cpu().numpy())
                    all_train_labels.extend(labels.cpu().numpy())
                    
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                    
                #scheduler.step()
            
            train_f1 = f1_score(all_train_labels, all_train_preds)
            train_acc = accuracy_score(all_train_labels, all_train_preds)
            train_f1_scores.append(train_f1)
            train_acc_scores.append(train_acc)

            # Evaluation
            model.eval()
            all_val_preds, all_val_labels = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    outputs = model(inputs)
                    preds = (outputs > 0.5).float()
                    all_val_preds.extend(preds.cpu().numpy())
                    all_val_labels.extend(labels.cpu().numpy())

            
            val_f1 = f1_score(all_val_labels, all_val_preds)
            val_acc = accuracy_score(all_val_labels, all_val_preds)
            val_f1_scores.append(val_f1)
            val_acc_scores.append(val_acc)
            
            print(f"Fold {fold+1}: Train F1={train_f1:.4f}, Val F1={val_f1:.4f}")

        config['avg_train_f1'] = np.mean(train_f1_scores)
        config['avg_val_f1'] = np.mean(val_f1_scores)
        config['avg_train_acc'] = np.mean(train_acc_scores)
        config['avg_val_acc'] = np.mean(val_acc_scores)
        
        print(f"✅ Config {config['hidden_dims']} → Train acc: {config['avg_train_acc']:.4f}, Val acc: {config['avg_val_acc']:.4f}")
        print(f"✅ Config {config['hidden_dims']} → Train F1: {config['avg_train_f1']:.4f}, Val F1: {config['avg_val_f1']:.4f}")

        results.append(config)

    return sorted(results, key=lambda x: x['avg_val_f1'], reverse=True)


In [5]:
def train_final_model(X_scaled, y, hidden_dims, dropout=0.3, num_epochs=100, batch_size=32, save_path="final_model.pt"):
    print("\n🚀 Training final model with best config...")
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

    model = MLP(input_size=X_scaled.shape[1], hidden_dims=hidden_dims, dropout=dropout)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    #scheduler = StepLR(optimizer, step_size=50, gamma=0.9)

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        #scheduler.step()
    torch.save(model.state_dict(), save_path)
    print(f"✅ Final model saved to {save_path}")
    return model


In [6]:
def evaluate_on_test(model, X_test, y_test):
    model.eval()
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    with torch.no_grad():
        outputs = model(X_test_tensor)
        preds = (outputs > 0.5).float().cpu().numpy()
        labels = y_test_tensor.cpu().numpy()

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    print(f"\n🧪 Final Evaluation on Test Set:")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")


In [7]:
model_configs = [
    {"hidden_dims": [16, 8, 4], "dropout": 0.3},
    {"hidden_dims": [32, 16], "dropout": 0.3},
    {"hidden_dims": [64, 32], "dropout": 0.2},
    {"hidden_dims": [16, 8], "dropout": 0.3}
]

# Step 1: Select the best model
sorted_configs = cross_validate_models(X_scaled, y, model_configs)


🔍 Testing model config: {'hidden_dims': [16, 8, 4], 'dropout': 0.3}
Fold 1: Train F1=0.9109, Val F1=1.0000
Fold 2: Train F1=0.9079, Val F1=1.0000
Fold 3: Train F1=0.8481, Val F1=1.0000
Fold 4: Train F1=0.9201, Val F1=1.0000
Fold 5: Train F1=0.9335, Val F1=1.0000
✅ Config [16, 8, 4] → Train acc: 0.9037, Val acc: 1.0000
✅ Config [16, 8, 4] → Train F1: 0.9041, Val F1: 1.0000

🔍 Testing model config: {'hidden_dims': [32, 16], 'dropout': 0.3}
Fold 1: Train F1=0.9764, Val F1=1.0000
Fold 2: Train F1=0.9728, Val F1=1.0000
Fold 3: Train F1=0.9709, Val F1=1.0000
Fold 4: Train F1=0.9802, Val F1=1.0000
Fold 5: Train F1=0.9765, Val F1=1.0000
✅ Config [32, 16] → Train acc: 0.9753, Val acc: 1.0000
✅ Config [32, 16] → Train F1: 0.9754, Val F1: 1.0000

🔍 Testing model config: {'hidden_dims': [64, 32], 'dropout': 0.2}
Fold 1: Train F1=0.9876, Val F1=1.0000
Fold 2: Train F1=0.9873, Val F1=1.0000
Fold 3: Train F1=0.9871, Val F1=1.0000
Fold 4: Train F1=0.9879, Val F1=1.0000
Fold 5: Train F1=0.9829, Val F1

In [8]:
best_config = sorted_configs[0]
print(f"best config: {best_config}")

# Step 2: retrain best model on full training set
final_model = train_final_model(X_scaled, y, hidden_dims=best_config['hidden_dims'], dropout=best_config['dropout'])

# Step 3: test set
X_test, y_test = input(test_set_bs)
evaluate_on_test(final_model, X_test, y_test)


best config: {'hidden_dims': [16, 8, 4], 'dropout': 0.3, 'avg_train_f1': 0.9040949188751071, 'avg_val_f1': 1.0, 'avg_train_acc': 0.9036687499999999, 'avg_val_acc': 1.0}

🚀 Training final model with best config...
✅ Final model saved to final_model.pt
(22, 50)
Best response
Label distribution: Counter({0.0: 11, 1.0: 11})

🧪 Final Evaluation on Test Set:
Accuracy: 0.8182, Precision: 0.8182, Recall: 0.8182, F1: 0.8182


In [9]:
'''

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    print(f"\n===== Fold {fold + 1} =====")

    # Split
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    #print(X_train.shape)
    y_train, y_val = y[train_idx], y[val_idx]

    # Convert to tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

    # DataLoaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    # Model
    model = MLP(input_size=X_train.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)

    for epoch in range(100):
        model.train()
        running_loss = 0.0
        all_train_preds, all_train_labels = [], []

        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            predictions = (outputs > 0.5).float()
            all_train_preds.extend(predictions.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

        model.eval()
        val_loss = 0.0
        all_val_preds, all_val_labels = [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                predictions = (outputs > 0.5).float()
                all_val_preds.extend(predictions.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        # 可加入 early stopping 條件（例如連續 X epoch val F1 沒提升）

    # 最後一個 epoch 結果（或 early stopping 結果）
    val_accuracy = accuracy_score(all_val_labels, all_val_preds)
    val_precision = precision_score(all_val_labels, all_val_preds)
    val_recall = recall_score(all_val_labels, all_val_preds)
    val_f1 = f1_score(all_val_labels, all_val_preds)

    print(f"Fold {fold + 1} Results:")
    print(f"  Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

    fold_results.append({
        'fold': fold + 1,
        'accuracy': val_accuracy,
        'precision': val_precision,
        'recall': val_recall,
        'f1': val_f1
    })

# 存下每個 fold 的結果
df_cv_results = pd.DataFrame(fold_results)
df_cv_results.to_csv("cv_results.csv", index=False)
'''

'\n\nskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n\nfold_results = []\n\nfor fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):\n    print(f"\n===== Fold {fold + 1} =====")\n\n    # Split\n    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]\n    #print(X_train.shape)\n    y_train, y_val = y[train_idx], y[val_idx]\n\n    # Convert to tensors\n    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)\n    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)\n    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)\n    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)\n\n    # DataLoaders\n    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)\n    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)\n    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)\n    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)\n\n    # Model\n    model = MLP(input_si